# Static Web Scraping 1.0

In [55]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

| Parser | Speed | Built-in | Best Use |
|---|---|---|---|
| html.parser | Medium | Yes | Beginners/general |
| lxml | Fast | No | Real scraping projects |
| html5lib | Slow | No | Broken HTML |
| xml | Fast | Depends | XML data |

### Page 1: Hockey Teams: Forms, Searching and Pagination
```
https://www.scrapethissite.com/pages/forms/

In [4]:
url = r"https://www.scrapethissite.com/pages/forms/"
response = requests.get(url)

In [5]:
soup = BeautifulSoup(response.text, "html")

# print(soup.prettify())

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Hockey Teams: Forms, Searching and Pagination | Scrape This Site | A public sandbox for learning web scraping
  </title>
  <link href="/static/images/scraper-icon.png" rel="icon" type="image/png"/>
  <meta content="width=device-width, initial-scale=1.0" name="viewport"/>
  <meta content="Browse through a database of NHL team stats since 1990. Practice building a scraper that handles common website interface components." name="description"/>
  <link crossorigin="anonymous" href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/css/bootstrap.min.css" integrity="sha256-MfvZlkHCEqatNoGiOXveE8FIwMzZg4W85qfrfIFBfYc= sha512-dTfge/zgoMYpP7QbHy4gWMEGsbsdZeCXz7irItjcC3sPUFtf0kuFbDz/ixG7ArTxmDjLXDmezHubeNikyKGVyQ==" rel="stylesheet"/>
  <link href="https://fonts.googleapis.com/css?family=Lato:400,700" rel="stylesheet" type="text/css"/>
  <link href="/static/css/styles.css" rel="stylesheet" type="text/css"/>
  <meta con

#### find() vs find_all()


- class="" → use class_
- id="" → use id
- for single attribute -> use href=""
- for custom attribute -> use attrs={"data-id": "12"})

```
soup.find(
    "a",
    attrs={
        "class": "btn",
        "href": "/login"
    }
)

```
```parent = soup.find(...) ```

```children = parent.find_all(...)```

In [34]:
nav = soup.find("nav", id="site-nav").find("a", href="/")
print(nav.text.strip())

Scrape This Site


In [32]:
nav_tabs = soup.find("ul", class_="nav-tabs").find_all("a")
texts = [link.text.strip() for link in nav_tabs[1:]]
print(texts)

['Sandbox', 'Lessons', 'FAQ', 'Login']


#### select_one() vs select()
``` Elements + Attributes + Tree Structure ```

soup.select_one("nav#site-nav div.container ul")
  ```
  nav with id=site-nav
    -> div with class=container
        -> ul
```
soup.select("li.item")
``` 
    Returns all matching elements.

In [33]:
nav = soup.select_one('nav#site-nav a[href="/"]')

print(nav.text.strip())

Scrape This Site


In [35]:
nav_tabs = soup.select("ul.nav-tabs a")
texts = [link.text.strip() for link in nav_tabs[1:]]

print(texts)

['Sandbox', 'Lessons', 'FAQ', 'Login']


### Page 2: List of the largest public / publicly traded companies
```
https://en.wikipedia.org/wiki/List_of_largest_companies_in_the_United_States_by_revenue

In [56]:
url = 'https://en.wikipedia.org/wiki/List_of_largest_companies_in_the_United_States_by_revenue'
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

In [57]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.prettify())

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" dir="ltr" lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   List of largest companies in the United States by revenue - Wikipedia
  </title>
  <script>
   (function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinn

In [90]:
table = soup.find_all('table')[0]

print(table.prettify())

<table class="wikitable sortable">
 <caption>
 </caption>
 <tbody>
  <tr>
   <th>
    Rank
   </th>
   <th>
    Name
   </th>
   <th>
    Industry
   </th>
   <th>
    Revenue
    <br/>
    (USD millions)
   </th>
   <th>
    Revenue growth
   </th>
   <th>
    Employees
   </th>
   <th>
    Headquarters
   </th>
  </tr>
  <tr>
   <td>
    1
   </td>
   <td>
    <a href="/wiki/Walmart" title="Walmart">
     Walmart
    </a>
   </td>
   <td>
    Retail
   </td>
   <td style="text-align:center;">
    680,985
   </td>
   <td style="text-align:center;">
    <span typeof="mw:File">
     <span title="Increase">
      <img alt="Increase" class="mw-file-element" data-file-height="300" data-file-width="300" decoding="async" height="11" src="//upload.wikimedia.org/wikipedia/commons/thumb/b/b0/Increase2.svg/20px-Increase2.svg.png?utm_source=en.wikipedia.org&amp;utm_campaign=parser&amp;utm_content=thumbnail" srcset="//upload.wikimedia.org/wikipedia/commons/thumb/b/b0/Increase2.svg/40px-Increase2.s

In [91]:
world_titles = table.find_all('th')
headers = [title.text.strip() for title in world_titles]

print(headers)

['Rank', 'Name', 'Industry', 'Revenue (USD millions)', 'Revenue growth', 'Employees', 'Headquarters']


In [92]:
df = pd.DataFrame(columns=headers)
df

,Rank,Name,Industry,Revenue (USD millions),Revenue growth,Employees,Headquarters


In [95]:
rows = table.find_all('tr')[1:]

for row in rows:
    cols = row.find_all('td')
    if cols:
        clean_row = [col.text.strip() for col in cols]
        df.loc[len(df)] = clean_row
        # print(clean_row)


In [96]:
df.head()

,Rank,Name,Industry,Revenue (USD millions),Revenue growth,Employees,Headquarters
0,1,Walmart,Retail,"680,985",5.1%,"2,100,000","Bentonville, Arkansas"
1,2,Amazon,Retail and cloud computing,"637,959",11.0%,"1,556,000","Seattle, Washington"
2,3,UnitedHealth Group,Healthcare,"400,278",7.7%,"400,000","Minnetonka, Minnesota"
3,4,Apple,Technology,"391,035",2.0%,"164,000","Cupertino, California"
4,5,CVS Health,Healthcare,"372,809",4.2%,"259,500","Woonsocket, Rhode Island"


In [100]:
df.to_csv(r"E:\Learning\Courses\@Youtube\Data Analyst\5. Python\Practice\Companies.csv", index=False)